# Day 1: PTB-XL arrhythmia classifier
Runtime > Change runtime type > **T4 GPU**, then Run all (~20-30 min).

In [2]:
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

Tesla T4


In [3]:
!nvidia-smi

Wed Oct  7 13:20:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
!pip -q install wfdb
if not os.path.exists('/content/MedTech-Suite'):
    !git clone -q https://github.com/aryan9-6-5/MedTech-Suite.git /content/MedTech-Suite
%cd /content/MedTech-Suite/day-1-ptbxl-arrhythmia-classifier


/content/MedTech-Suite/day-1-ptbxl-arrhythmia-classifier


In [5]:
# 100 Hz records + metadata from PhysioNet's public S3 mirror (~0.8 GB, parallel)
!pip -q install awscli
# Boost AWS S3 concurrency from default 10 to 64 threads (~1.5-2 mins instead of ~13 mins)
!aws configure set default.s3.max_concurrent_requests 64
!mkdir -p data
!aws s3 sync --no-sign-request --only-show-errors s3://physionet-open/ptb-xl/1.0.3/records100 data/records100
!aws s3 cp --no-sign-request --only-show-errors s3://physionet-open/ptb-xl/1.0.3/ptbxl_database.csv data/
!aws s3 cp --no-sign-request --only-show-errors s3://physionet-open/ptb-xl/1.0.3/scp_statements.csv data/
!find data/records100 -name '*.dat' | wc -l   # expect 21799


21799


In [6]:
%cd src
!python train.py

/content/MedTech-Suite/day-1-ptbxl-arrhythmia-classifier/src
epoch 01 val macro AUC 0.8882
Traceback (most recent call last):
  File "/content/MedTech-Suite/day-1-ptbxl-arrhythmia-classifier/src/train.py", line 84, in <module>
    main()
    ~~~~^^
  File "/content/MedTech-Suite/day-1-ptbxl-arrhythmia-classifier/src/train.py", line 66, in main
    torch.save(model.state_dict(), ROOT / "results" / "best.pt")
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1003, in save
    with _open_zipfile_writer(f) as opened_zipfile:
         ~~~~~~~~~~~~~~~~~~~~^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 865, in _open_zipfile_writer
    return container(name_or_buffer)  # type: ignore[arg-type]
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 829, in __init__
    torch._C.PyTorchFileWriter(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^
        self.name,

In [7]:
!python evaluate.py

Traceback (most recent call last):
  File "/content/MedTech-Suite/day-1-ptbxl-arrhythmia-classifier/src/evaluate.py", line 72, in <module>
    main()
    ~~~~^^
  File "/content/MedTech-Suite/day-1-ptbxl-arrhythmia-classifier/src/evaluate.py", line 21, in main
    idx = np.load(R / "test_idx.npy")
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 459, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
                              ~~~~^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/MedTech-Suite/day-1-ptbxl-arrhythmia-classifier/results/test_idx.npy'


In [8]:

# Show results inline so they are saved in this notebook's outputs (works in VS Code and browser Colab)
import json
from IPython.display import Image, display
print(open('results/metrics.json').read()[:1500]); print(open('results/f1_at_0.5.json').read())
display(Image('results/roc.png')); display(Image('results/saliency_mi.png'))
try:
    !zip -qr results.zip results/metrics.json results/f1_at_0.5.json results/roc.png results/saliency_mi.png
    from google.colab import files; files.download('results.zip')   # browser Colab only
except Exception as e: print('download skipped:', e)


FileNotFoundError: [Errno 2] No such file or directory: 'results/metrics.json'